# NDC + Active Drug Lookup

In [1]:
ndc = '00008084381'

In [2]:
import pandas as pd
import requests

# ndc = '00008084381'
url = f"http://localhost:4000/REST/ndcproperties.json?id={ndc}"

response = requests.get(url)


if response.status_code == 200:
    json_data = response.json()
    ndc_properties = json_data['ndcPropertyList']['ndcProperty']
    
    # Normalize just the main level first
    df_base = pd.json_normalize(ndc_properties)
    
    # Extract packaging
    df_base['packaging'] = df_base['packagingList.packaging'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
    )
    
    # Now normalize properties
    df_properties = pd.json_normalize(
        ndc_properties,
        record_path=['propertyConceptList', 'propertyConcept'],
        meta=['ndcItem', 'rxcui', 'source'],
        errors='ignore'
    )
    
    # Pivot
    df_wide = df_properties.pivot_table(
        index=['ndcItem', 'rxcui', 'source'],
        columns='propName',
        values='propValue',
        aggfunc='first'
    ).reset_index()
    
    # Merge with packaging
    df_final = df_wide.merge(
        df_base[['ndcItem', 'rxcui', 'source', 'packaging']], 
        on=['ndcItem', 'rxcui', 'source']
    )
    
    display(df_final)

,ndcItem,rxcui,source,COLOR,COLORTEXT,IMPRINT_CODE,LABELER,LABEL_TYPE,MARKETING_CATEGORY,MARKETING_EFFECTIVE_TIME_LOW,MARKETING_STATUS,NDA,SCORE,SHAPE,SHAPETEXT,SIZE,packaging
0,00008084381,251872,FDA,NaN,NaN,NaN,"Wyeth Pharmaceuticals LLC, a subsidiary of Pfi...",HUMAN PRESCRIPTION DRUG,NDA,20000501,ACTIVE,NDA020987,NaN,NaN,NaN,NaN,"90 TABLET, DELAYED RELEASE in 1 BOTTLE (0008-0..."
1,00008084381,352125,Hybrid,C48330,YELLOW(YELLOW),P;20,"Wyeth Pharmaceuticals LLC, a subsidiary of Pfi...",HUMAN PRESCRIPTION DRUG,NDA,20000501,ACTIVE,NDA020987,1,C48345,BICONVEX,9 mm,"90 TABLET, DELAYED RELEASE in 1 BOTTLE (0008-0..."


In [3]:
# 1. Get all unique RxCUIs as strings
unique_rxcuis = df_final['rxcui'].astype(str).unique()

active_drug_results = []

# 2. Loop through each RxCUI
for rxcui in unique_rxcuis:
    active_url = f"http://localhost:4000/REST/rxcui/{rxcui}/active.json"
    
    try:
        response = requests.get(active_url)
        if response.status_code == 200:
            json_data = response.json()
            
            # Safe access using .get() to avoid crashing if keys are missing
            group = json_data.get('minConceptGroup', {})
            concepts = group.get('minConcept', [])
            
            if concepts:
                # Usually there is one active drug, but this handles if multiple return
                for concept in concepts:
                    active_drug_results.append({
                        'rxcui': rxcui, # Keep this key for merging later
                        'active_name': concept.get('name'),
                        'active_tty': concept.get('tty')
                    })
            else:
                # Record that we checked but found nothing
                active_drug_results.append({
                    'rxcui': rxcui, 
                    'active_name': 'No Active Drug Found',
                    'active_tty': None
                })
                
    except Exception as e:
        print(f"Error processing RxCUI {rxcui}: {e}")

# 3. Create a DataFrame from the results
df_active = pd.DataFrame(active_drug_results)

# 4. Merge back to your main dataframe
# We use 'rxcui' as the key. We convert to string to ensure types match.
df_final['rxcui'] = df_final['rxcui'].astype(str)
df_active['rxcui'] = df_active['rxcui'].astype(str)

final_combined_df = df_final.merge(df_active, on='rxcui', how='left')

# Display specific columns to verify
display(final_combined_df[['ndcItem', 'rxcui', 'active_name']])

,ndcItem,rxcui,active_name
0,00008084381,251872,pantoprazole 20 MG Delayed Release Oral Tablet
1,00008084381,352125,pantoprazole 20 MG Delayed Release Oral Tablet...
